# Lesson 02 - Ground agents with Python tools

**Level:** beginner/intermediate | **Demo time:** 10 minutes | **Builds on:** Lesson 01

Tools let a model request deterministic application behavior. MAF derives a tool schema from Python type hints and docstrings, sends the available definitions to the model, executes selected functions, and returns results to the model for a final answer.

## Learning objectives

- Define tool contracts with type annotations and descriptions.
- Give an agent access only to the tools it needs.
- Distinguish model reasoning from trusted system operations.

## The tool-calling loop

```text
Customer question
  -> model chooses a tool and arguments
  -> MAF validates and invokes Python
  -> tool result returns to the model
  -> model explains the result to the customer
```

The model does **not** execute arbitrary Python. It can only request tools registered by the application.

In [ ]:
from maf_foundry_hackathon.support_tools import lookup_order, search_support_policy

for support_tool in (lookup_order, search_support_policy):
    print(f"{support_tool.name}: {support_tool.description}")

These tools use simulated workshop data, so rerunning the demo is safe. They are marked `approval_mode="never_require"` because they only read deterministic local data. A real refund, cancellation, or purchase should require explicit authorization and usually human approval.

In [ ]:
from agent_framework import Agent

from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)

settings = WorkshopSettings.from_env()
credential = create_credential()

grounded_agent = Agent(
    client=create_chat_client(settings, credential),
    name="grounded_support_agent",
    description="Answers order and policy questions using approved support tools.",
    instructions=(
        "You are a Contoso support agent. Use tools whenever a question depends on "
        "an order or policy. Clearly distinguish verified facts from recommendations. "
        "Never claim that you changed an order."
    ),
    tools=[lookup_order, search_support_policy],
)

In [ ]:
question = (
    "Order ORD-1001 arrived with a cracked ear cup. "
    "Can I return it, and who pays for shipping?"
)
response = await grounded_agent.run(question)

print("Customer:", question)
print("Agent:", response.text)

## What to point out during the demo

1. The prompt contains an order ID and a policy question.
2. The agent can choose both tools; the notebook does not hard-code the sequence.
3. The final answer combines verified order data with the return policy.
4. The tools return strings, but production tools can return typed JSON and can call APIs or databases.

> Presenter prompt: change `ORD-1001` to `ORD-9999` and show how a tool's explicit failure becomes a useful clarification instead of fabricated data.

In [ ]:
missing_order_response = await grounded_agent.run(
    "Please tell me whether order ORD-9999 can still be canceled."
)
print(missing_order_response.text)

## Production design checklist

- Keep tool inputs narrow and strongly typed.
- Validate authorization inside the tool; never trust the model as the security boundary.
- Return explicit errors rather than success-shaped fallback data.
- Require approval for consequential writes.
- Log tool name, duration, outcome, and a correlation ID without logging sensitive payloads.

In [ ]:
credential.close()

## Recap

Tools ground an agent in trusted operations. Lesson 03 makes the agent's *output* equally dependable by validating it against a Pydantic schema.